**Import des librairies :**

In [1]:
from pyspark.sql.functions import (
    col, when, lit, avg, count, min, max,
    round as spark_round,
    sum as spark_sum,
    monotonically_increasing_id
)
from pyspark.sql.types import IntegerType

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 3, Finished, Available, Finished, False)

**Chargement Silver :**


In [2]:
df = spark.read.format("delta").table("silver_supply_chain")
print(f"Silver chargé : {df.count()} lignes x {len(df.columns)} colonnes")
display(df.limit(3))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 4, Finished, Available, Finished, False)

Silver chargé : 100 lignes x 33 colonnes


SynapseWidget(Synapse.DataFrame, 0c8ced72-2336-4163-8a94-abb06234f670)

**Dimension Produit :**

In [3]:
dim_product = df.select(
    "sku",
    "product_type",
    "price",
    "availability"
).dropDuplicates(["sku"])

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_product")

print(f"gold_dim_product : {dim_product.count()} lignes")
display(dim_product.limit(3))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 5, Finished, Available, Finished, False)

gold_dim_product : 100 lignes


SynapseWidget(Synapse.DataFrame, 42bf09a9-036e-40d2-8a14-62dce849c005)

**Dimension Fournisseur :**

In [4]:
dim_supplier = df.select(
    "supplier_name",
    "location",
    "production_volumes"
).dropDuplicates(["supplier_name"])

dim_supplier.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_supplier")

print(f"gold_dim_supplier : {dim_supplier.count()} lignes")
display(dim_supplier.limit(3))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 6, Finished, Available, Finished, False)

gold_dim_supplier : 5 lignes


SynapseWidget(Synapse.DataFrame, 84eb63a9-f7b0-437a-a15d-638972cfdc2a)

**Dimension Transport :**

In [5]:
dim_shipping = df.select(
    "shipping_carriers",
    "transportation_modes",
    "routes"
).dropDuplicates(["shipping_carriers", "transportation_modes", "routes"])

dim_shipping.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_shipping")

print(f"gold_dim_shipping : {dim_shipping.count()} lignes")
display(dim_shipping.limit(3))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 7, Finished, Available, Finished, False)

gold_dim_shipping : 33 lignes


SynapseWidget(Synapse.DataFrame, 283d76e6-1cbb-455c-a3ac-c2d6fdc9d602)

**Décision customer_demographics :**

In [6]:
# ============================================================
# DÉCISION MODÉLISATION — customer_demographics
# ============================================================
# Valeurs observées : Male, Female, Non-binary
# → Variable catégorielle agrégée, pas d'identifiant client unique
# → Pas de dim_customer justifiable (pas d'attributs supplémentaires)
# → Intégrée comme attribut de segmentation dans fact_orders
#    pour permettre le filtrage direct dans Power BI
#    sans jointure supplémentaire
# ============================================================
print("customer_demographics : attribut de segmentation → intégré dans fact_orders.")

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 8, Finished, Available, Finished, False)

customer_demographics : attribut de segmentation → intégré dans fact_orders.


**Dimension Inspection Qualité :**

In [7]:
dim_inspection = df.select(
    "sku",
    "inspection_results",
    "defect_rates",
    "quality_flag"
).dropDuplicates(["sku"])

dim_inspection.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_inspection")

print(f"gold_dim_inspection : {dim_inspection.count()} lignes")
display(dim_inspection.limit(3))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 9, Finished, Available, Finished, False)

gold_dim_inspection : 100 lignes


SynapseWidget(Synapse.DataFrame, a5dfb332-8fb9-4071-9ac1-997a5363f40d)

**Table de Faits centrale :**

In [8]:
# ============================================================
# GRAIN DE LA TABLE DE FAITS
# ============================================================
# 1 ligne = 1 SKU unique (dataset agrégé, 100 SKUs distincts)
# Pas de dimension temporelle → analyses comparatives cross-produit
# customer_demographics intégré comme attribut de segmentation
# ============================================================

fact_orders = df.select(
    # Clés de jointure vers les dimensions
    "sku",
    "supplier_name",
    "shipping_carriers",
    "transportation_modes",
    "routes",
    "product_type",

    # Segmentation client (attribut catégoriel, pas de dim_customer)
    "customer_demographics",

    # Métriques de volume
    "number_of_products_sold",
    "order_quantities",
    "stock_levels",
    "production_volumes",

    # Métriques financières
    "revenue_generated",
    "manufacturing_costs",
    "shipping_costs",
    "costs",
    "price",
    "profit_margin_pct",

    # Métriques de délai
    "inbound_lead_time",
    "manufacturing_lead_time",
    "outbound_lead_time",
    "shipping_times",
    "end_to_end_lead_time",
    "inbound_lead_time_pct",
    "manufacturing_lead_time_pct",
    "outbound_lead_time_pct",

    # Métriques qualité
    "defect_rates",
    "inspection_results",

    # KPIs dérivés
    "stock_coverage",
    "bottleneck_segment",
    "e2e_delay_flag",
    "quality_flag"
)

fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_orders")

print(f"gold_fact_orders : {fact_orders.count()} lignes x {len(fact_orders.columns)} colonnes")
display(fact_orders.limit(5))

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 10, Finished, Available, Finished, False)

gold_fact_orders : 100 lignes x 31 colonnes


SynapseWidget(Synapse.DataFrame, a99f4e2e-9df2-4097-b270-128f5ca50969)

**Validation Gold (contrôle qualité) :**

In [9]:
print("=== Contrôle qualité Gold ===\n")

tables = {
    "gold_fact_orders"   : fact_orders,
    "gold_dim_product"   : dim_product,
    "gold_dim_supplier"  : dim_supplier,
    "gold_dim_shipping"  : dim_shipping,
    "gold_dim_inspection": dim_inspection
}

print("--- Dimensions des tables ---")
for name, tbl in tables.items():
    print(f"  {name} : {tbl.count()} lignes x {len(tbl.columns)} colonnes")

print("\n--- Nulls dans fact_orders ---")
display(
    fact_orders.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in fact_orders.columns
    ])
)

print("\n--- KPIs de synthèse par product_type ---")
display(
    fact_orders.groupBy("product_type").agg(
        count("sku").alias("nb_skus"),
        spark_round(spark_sum("revenue_generated"), 2).alias("total_revenue"),
        spark_round(avg("profit_margin_pct"), 2).alias("avg_margin_pct"),
        spark_round(avg("end_to_end_lead_time"), 1).alias("avg_e2e_lead_time"),
        spark_round(avg("defect_rates"), 4).alias("avg_defect_rate"),
        spark_round(avg("stock_coverage"), 2).alias("avg_stock_coverage")
    ).orderBy("total_revenue", ascending=False)
)

print("\n--- Performance par transporteur ---")
display(
    fact_orders.groupBy("shipping_carriers").agg(
        spark_round(avg("outbound_lead_time"), 1).alias("avg_outbound_lead_time"),
        spark_round(avg("shipping_costs"), 2).alias("avg_shipping_cost"),
        spark_round(avg("defect_rates"), 4).alias("avg_defect_rate"),
        count("sku").alias("nb_skus")
    ).orderBy("avg_outbound_lead_time")
)

print("\n--- Distribution des goulots d'étranglement ---")
display(
    fact_orders.groupBy("bottleneck_segment", "product_type")
        .count()
        .orderBy("product_type", "count", ascending=False)
)

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 11, Finished, Available, Finished, False)

=== Contrôle qualité Gold ===

--- Dimensions des tables ---
  gold_fact_orders : 100 lignes x 31 colonnes
  gold_dim_product : 100 lignes x 4 colonnes
  gold_dim_supplier : 5 lignes x 3 colonnes
  gold_dim_shipping : 33 lignes x 3 colonnes
  gold_dim_inspection : 100 lignes x 4 colonnes

--- Nulls dans fact_orders ---


SynapseWidget(Synapse.DataFrame, f26f5181-3287-46d9-a99a-58ff55bf76fe)


--- KPIs de synthèse par product_type ---


SynapseWidget(Synapse.DataFrame, 4d66ef0b-e341-4051-8fd1-5e1e81d7658c)


--- Performance par transporteur ---


SynapseWidget(Synapse.DataFrame, 88c02c22-6dc1-4d85-af86-3c793e1eca59)


--- Distribution des goulots d'étranglement ---


SynapseWidget(Synapse.DataFrame, dab1c51b-5cba-4a7e-948a-e7f819aeb569)

**Vérification intégrité référentielle :**

In [10]:
print("=== Vérification intégrité référentielle ===\n")

orphan_skus = fact_orders.join(
    dim_product, on="sku", how="left_anti"
).count()
print(f"SKUs orphelins (fact sans dim_product) : {orphan_skus}")

orphan_suppliers = fact_orders.join(
    dim_supplier, on="supplier_name", how="left_anti"
).count()
print(f"Fournisseurs orphelins (fact sans dim_supplier) : {orphan_suppliers}")

orphan_shipping = fact_orders.join(
    dim_shipping,
    on=["shipping_carriers", "transportation_modes", "routes"],
    how="left_anti"
).count()
print(f"Transporteurs orphelins (fact sans dim_shipping) : {orphan_shipping}")

orphan_inspection = fact_orders.join(
    dim_inspection, on="sku", how="left_anti"
).count()
print(f"SKUs orphelins (fact sans dim_inspection) : {orphan_inspection}")

if all(x == 0 for x in [orphan_skus, orphan_suppliers, orphan_shipping, orphan_inspection]):
    print("\nIntégrité référentielle : OK — toutes les clés sont cohérentes.")
else:
    print("\nAttention : des orphelins ont été détectés, vérifier les jointures.")

StatementMeta(, fb6a7c63-3aa0-4742-ac60-c512b8b3fc90, 12, Finished, Available, Finished, False)

=== Vérification intégrité référentielle ===

SKUs orphelins (fact sans dim_product) : 0
Fournisseurs orphelins (fact sans dim_supplier) : 0
Transporteurs orphelins (fact sans dim_shipping) : 0
SKUs orphelins (fact sans dim_inspection) : 0

Intégrité référentielle : OK — toutes les clés sont cohérentes.
